In [3]:
from langchain.document_loaders.csv_loader import CSVLoader

In [2]:
!pip install -qU langchain langchain-community langchain-openai langchain-pinecone pinecone-client

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandasai 0.2.12 requires openai<0.28.0,>=0.27.5, but you have openai 1.69.0 which is incompatible.


In [6]:
loader = CSVLoader(file_path="/Users/amogh/Documents/amogh/personal/cohere_hack/data/prompts_db.csv")

data = loader.load()
data

[Document(metadata={'source': '/Users/amogh/Documents/amogh/personal/cohere_hack/data/prompts_db.csv', 'row': 0}, page_content="act: SEO Prompt\nprompt: Using WebPilot, create an outline for an article that will be 2,000 words on the keyword 'Best SEO prompts' based on the top 10 results from Google. Include every relevant heading possible. Keep the keyword density of the headings high. For each section of the outline, include the word count. Include FAQs section in the outline too, based on people also ask section from Google for the keyword. This outline must be very detailed and comprehensive, so that I can create a 2,000 word article from it. Generate a long list of LSI and NLP keywords related to my keyword. Also include any other words related to the keyword. Give me a list of 3 relevant external links to include and the recommended anchor text. Make sure they’re not competing articles. Split the outline into part 1 and part 2."),
 Document(metadata={'source': '/Users/amogh/Docum

In [3]:
import os
from langchain.embeddings import CohereEmbeddings
# embeddings = CohereEmbeddings(cohere_api_key=os.environ['COHERE_API_KEY'], model="embed-english-light-v3.0")
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [8]:
from langchain.vectorstores import Pinecone
import pinecone
import os
# initialize pinecone
pinecone.init(
    api_key=os.environ.get('PINECONE_API_KEY'),  # find at app.pinecone.io
    environment='us-west1-gcp',  # next to api key in console,
)

index_name = "prompts"

docsearch = Pinecone.from_documents(data, embeddings, index_name=index_name)

In [ ]:
from langchain.vectorstores import Pinecone
import pinecone
import os

os.environ["PINECONE_API_KEY"] = "" # Replace with your actual API key


# initialize pinecone using Pinecone class
pc = pinecone.Pinecone(
    api_key='',  # get from environment variable
    environment="us-east-1"  # replace with your environment
)

/opt/homebrew/Caskroom/miniforge/base/envs/langchain/lib/python3.11/site-packages/pinecone/data/index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
import time

index_name = "spark-prompts"  # change if desired

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

In [5]:
index = pc.Index(index_name)

from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [12]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(data))]

vector_store.add_documents(documents=data, ids=uuids)

['712498bc-3b38-40a7-9aea-1ff1e676efa4',
 '88f548a8-c89b-4110-b556-438e9fd5fda8',
 '2ddd4d12-2e47-4494-85eb-b28db3a6aaba',
 '801814e8-c54d-4334-a0da-401a0685f0ee',
 '2dfd9556-0999-480e-808e-372596a81e39',
 '0deb4599-f836-455a-b746-3ce15ae34515',
 'b21571c4-cd7d-4b44-a649-79895f80dc33',
 '2ece4a72-16fd-47b5-9a62-a2f3f4702d9d',
 'e7254b64-b826-4f4e-a361-44717d17bf11',
 'e431aed9-c80b-44a0-bb29-b49c1ab579bf',
 '524238dc-d1d0-4b16-bcf8-33f45e8e3382',
 '0142f11e-1821-4acb-82ad-9e5656d09b00',
 'a4a24038-45ab-4634-8f2b-3eabe1b76e47',
 '692cfd0a-db33-4748-824f-255d5c4991d1',
 '1511bf93-f808-498d-b427-bc98046a251d',
 '21d291bc-a062-4adb-ae70-3051cddb07c0',
 '051de6b3-63a0-4030-b85a-7c3e24ad82ea',
 'b3bf766b-95b6-425d-bfc9-c548e56a9250',
 '2bd3faf9-2bfc-4117-8565-265df7d45d51',
 '277556fd-72c0-48ba-9e95-50675dea7b14',
 'fd650c84-fd2c-4a24-a473-7be6e9af18e7',
 'd93f65cf-67a8-4ff8-8663-3c9762dc5855',
 'd381f037-aecc-491d-925e-aea11168b705',
 'ef94edf7-1ee3-48c3-ba03-aac3a3aa6da6',
 '3a6bdefa-00e2-

In [9]:
docsearch.similarity_search('seo prompt')

[Document(page_content="act: SEO Prompt\nprompt: Using WebPilot, create an outline for an article that will be 2,000 words on the keyword 'Best SEO prompts' based on the top 10 results from Google. Include every relevant heading possible. Keep the keyword density of the headings high. For each section of the outline, include the word count. Include FAQs section in the outline too, based on people also ask section from Google for the keyword. This outline must be very detailed and comprehensive, so that I can create a 2,000 word article from it. Generate a long list of LSI and NLP keywords related to my keyword. Also include any other words related to the keyword. Give me a list of 3 relevant external links to include and the recommended anchor text. Make sure they’re not competing articles. Split the outline into part 1 and part 2.", metadata={'row': 0.0, 'source': 'prompts.csv'}),
 Document(page_content='act: Midjourney Prompt Generator\nprompt: I want you to act as a prompt generator

In [6]:
# !pip install "playwright"
# !pip install "unstructured"
!pip install bs4 beautifulsoup

  Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
  Using cached BeautifulSoup-3.2.2.tar.gz (32 kB)
  Preparing metadata (setup.py) ... error
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [7 lines of output]
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 34, in <module>
        File "/private/var/folders/vm/xh0qc0y959s349y0mqr27vr00000gn/T/pip-install-tr15f6ec/beautifulsoup_a6002f5152f64ece9dd46fbd737d58f0/setup.py", line 3
          "You're trying to run a very old release of Beautiful Soup under Python 3. This will not work."<>"Please use Beautiful Soup 4, available through the pip package 'beautifulsoup4'."
                                                                                                         ^^
      SyntaxError: invalid syntax
      [end of output]
  
  note: This error originates from a subprocess, and 

In [11]:
!playwright install

131.1 Mb [                    ] 0% 0.0s131.1 Mb [                    ] 0% 76.2s131.1 Mb [                    ] 0% 87.7s131.1 Mb [                    ] 0% 96.2s131.1 Mb [                    ] 0% 98.1s131.1 Mb [                    ] 0% 99.3s131.1 Mb [                    ] 0% 100.7s131.1 Mb [                    ] 0% 100.1s131.1 Mb [                    ] 0% 100.6s131.1 Mb [                    ] 0% 101.7s131.1 Mb [                    ] 0% 101.2s131.1 Mb [                    ] 0% 106.4s131.1 Mb [                    ] 0% 105.3s131.1 Mb [                    ] 0% 106.6s131.1 Mb [                    ] 0% 107.9s131.1 Mb [                    ] 0% 107.1s131.1 Mb [                    ] 0% 102.8s131.1 Mb [                    ] 0% 101.2s131.1 Mb [                    ] 0% 99.6s131.1 Mb [                    ] 0% 96.9s131.1 Mb [                    ] 0% 93.8s131.1 Mb [                    ] 0% 92.3s131.1 Mb [                    ] 0% 91.1s131.1 Mb [                    ] 0% 90.3s131.1 Mb [                   

In [17]:
from langchain.document_loaders import PlaywrightURLLoader

In [7]:
urls = [
"https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
"https://github.com/brexhq/prompt-engineering",
"http://promptingguide.ai//",
"http://promptingguide.ai//course",
"http://promptingguide.ai//introduction",
"http://promptingguide.ai//introduction/settings",
"http://promptingguide.ai//introduction/basics",
"http://promptingguide.ai//introduction/elements",
"http://promptingguide.ai//introduction/tips",
"http://promptingguide.ai//introduction/examples",
"http://promptingguide.ai//techniques",
"http://promptingguide.ai//techniques/zeroshot",
"http://promptingguide.ai//techniques/fewshot",
"http://promptingguide.ai//techniques/cot",
"http://promptingguide.ai//techniques/consistency",
"http://promptingguide.ai//techniques/knowledge",
"http://promptingguide.ai//techniques/tot",
"http://promptingguide.ai//techniques/rag",
"http://promptingguide.ai//techniques/art",
"http://promptingguide.ai//techniques/ape",
"http://promptingguide.ai//techniques/activeprompt",
"http://promptingguide.ai//techniques/dsp",
"http://promptingguide.ai//techniques/react",
"http://promptingguide.ai//techniques/multimodalcot",
"http://promptingguide.ai//techniques/graph",
"http://promptingguide.ai//applications",
"http://promptingguide.ai//applications/pal",
"http://promptingguide.ai//applications/generating",
"http://promptingguide.ai//applications/coding",
"http://promptingguide.ai//applications/workplace_casestudy",
"http://promptingguide.ai//applications/pf",
"http://promptingguide.ai//models",
"http://promptingguide.ai//models/flan",
"http://promptingguide.ai//models/chatgpt",
"http://promptingguide.ai//models/llama",
"http://promptingguide.ai//models/gpt-4",
"http://promptingguide.ai//models/collection",
"http://promptingguide.ai//risks",
"http://promptingguide.ai//risks/adversarial",
"http://promptingguide.ai//risks/factuality",
"http://promptingguide.ai//risks/biases",
"http://promptingguide.ai//papers",
"http://promptingguide.ai//tools",
"http://promptingguide.ai//notebooks",
"http://promptingguide.ai//datasets",
"http://promptingguide.ai//readings",
"https://learnprompting.org/blog/2022/12/20/prompt-injection-competition",
"https://learnprompting.org/docs/additional",
"https://learnprompting.org/docs/advanced_applications/mrkl",
"https://learnprompting.org/docs/advanced_applications/overview",
"https://learnprompting.org/docs/advanced_applications/pal",
"https://learnprompting.org/docs/advanced_applications/react",
"https://learnprompting.org/docs/applied_prompting/build_chatbot_from_kb",
"https://learnprompting.org/docs/applied_prompting/build_chatgpt",
"https://learnprompting.org/docs/applied_prompting/mc_tutorial",
"https://learnprompting.org/docs/applied_prompting/overview",
"https://learnprompting.org/docs/applied_prompting/short_response",
"https://learnprompting.org/docs/basic_applications/blog_generation",
"https://learnprompting.org/docs/basic_applications/coding_assistance",
"https://learnprompting.org/docs/basic_applications/contracts",
"https://learnprompting.org/docs/basic_applications/emojis",
"https://learnprompting.org/docs/basic_applications/introduction",
"https://learnprompting.org/docs/basic_applications/study_tool",
"https://learnprompting.org/docs/basic_applications/summarize",
"https://learnprompting.org/docs/basic_applications/table_generation",
"https://learnprompting.org/docs/basic_applications/writing_emails",
"https://learnprompting.org/docs/basic_applications/writing_in_diff_voices",
"https://learnprompting.org/docs/basic_applications/zapier_for_emails",
"https://learnprompting.org/docs/basics/chatbot_basics",
"https://learnprompting.org/docs/basics/combining_techniques",
"https://learnprompting.org/docs/basics/configuration_hyperparameters",
"https://learnprompting.org/docs/basics/few_shot",
"https://learnprompting.org/docs/basics/formalizing",
"https://learnprompting.org/docs/basics/instructions",
"https://learnprompting.org/docs/basics/intro",
"https://learnprompting.org/docs/basics/pitfalls",
"https://learnprompting.org/docs/basics/prompting",
"https://learnprompting.org/docs/basics/roles",
"https://learnprompting.org/docs/basics/starting_your_journey",
"https://learnprompting.org/docs/basics/world",
"https://learnprompting.org/docs/bibliography",
"https://learnprompting.org/docs/category/-advanced-applications",
"https://learnprompting.org/docs/category/-applied-prompting",
"https://learnprompting.org/docs/category/-basic-applications",
"https://learnprompting.org/docs/category/-basics",
"https://learnprompting.org/docs/category/-defensive-measures",
"https://learnprompting.org/docs/category/-miscellaneous",
"https://learnprompting.org/docs/category/-offensive-measures",
"https://learnprompting.org/docs/category/-prompt-hacking",
"https://learnprompting.org/docs/category/-prompt-tuning",
"https://learnprompting.org/docs/category/-tooling",
"https://learnprompting.org/docs/category/prompt-engineering-ides",
"https://learnprompting.org/docs/hot_topics",
"https://learnprompting.org/docs/Images/fix_deformed_generations",
"https://learnprompting.org/docs/Images/intro",
"https://learnprompting.org/docs/Images/midjourney",
"https://learnprompting.org/docs/Images/quality_boosters",
"https://learnprompting.org/docs/Images/repetition",
"https://learnprompting.org/docs/Images/resources",
"https://learnprompting.org/docs/Images/style_modifiers",
"https://learnprompting.org/docs/Images/weighted_terms",
"https://learnprompting.org/docs/intermediate/chain_of_thought",
"https://learnprompting.org/docs/intermediate/generated_knowledge",
"https://learnprompting.org/docs/intermediate/least_to_most",
"https://learnprompting.org/docs/intermediate/self_consistency",
"https://learnprompting.org/docs/intermediate/whats_in_a_prompt",
"https://learnprompting.org/docs/intermediate/zero_shot_cot",
"https://learnprompting.org/docs/intro",
"https://learnprompting.org/docs/miscl/detect",
"https://learnprompting.org/docs/miscl/music",
"https://learnprompting.org/docs/miscl/trickery",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/filtering",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/instruction",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/llm_eval",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/other",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/overview",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/post_prompting",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/random_sequence",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/sandwich_defense",
"https://learnprompting.org/docs/prompt_hacking/defensive_measures/xml_tagging",
"https://learnprompting.org/docs/prompt_hacking/injection",
"https://learnprompting.org/docs/prompt_hacking/intro",
"https://learnprompting.org/docs/prompt_hacking/jailbreaking",
"https://learnprompting.org/docs/prompt_hacking/leaking",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/code_injection",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/defined_dictionary",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/indirect_injection",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/obfuscation",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/overview",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/payload_splitting",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/recursive_attack",
"https://learnprompting.org/docs/prompt_hacking/offensive_measures/virtualization",
"https://learnprompting.org/docs/reliability/calibration",
"https://learnprompting.org/docs/reliability/debiasing",
"https://learnprompting.org/docs/reliability/ensembling",
"https://learnprompting.org/docs/reliability/intro",
"https://learnprompting.org/docs/reliability/lm_self_eval",
"https://learnprompting.org/docs/reliability/math",
"https://learnprompting.org/docs/tooling/tools",
"https://learnprompting.org/docs/trainable/discretized",
"https://learnprompting.org/docs/trainable/soft_prompting"
]
len(urls)

141

In [8]:
import nest_asyncio
nest_asyncio.apply()

In [11]:
!pip install beautifulsoup4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 186.0/186.0 kB 352.0 kB/s eta 0:00:0000:0100:01


In [101]:
lil_urls = []

In [12]:
from langchain.document_loaders import WebBaseLoader
loader = WebBaseLoader(urls)
data = loader.load()
data

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en'}, page_content='\n\n\n\n\n\nPrompt Engineering | Lil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil\'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n

In [13]:
from langchain.text_splitter import TokenTextSplitter
text_splitter = TokenTextSplitter(chunk_size=512, chunk_overlap=25)
docs = text_splitter.split_documents(data)
docs[:10]

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en'}, page_content="\n\n\n\n\n\nPrompt Engineering | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n

In [136]:
import tiktoken
encoding_name = tiktoken.get_encoding("cl100k_base")
def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [43]:
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en', 'text': "\n\n\n\n\n\nPrompt Engineering | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPo

In [139]:
from langchain.embeddings import CohereEmbeddings
embeddings = CohereEmbeddings(model='embed-english-light-v3.0')

In [141]:
from langchain.vectorstores import Pinecone
import pinecone

# initialize pinecone
pinecone.init(
    api_key=os.environ['PINECONE_API_KEY'],  # find at app.pinecone.io
    environment='us-west1-gcp',  # next to api key in console,
)

index_name = "sparklearn"

docsearch = Pinecone.from_documents(docs, embeddings, index_name=index_name)

In [142]:
# if you already have an index, you can load it like this
docsearch = Pinecone.from_existing_index(index_name, embeddings)

query = "What is prompting?"
res = docsearch.similarity_search(query, k=5)
retriever = docsearch.as_retriever()
res

[Document(page_content=" Prompting, which applies the main concept from ToT frameworks as a simple prompting technique, getting the LLM to evaluate intermediate thoughts in a single prompt. A sample ToT prompt is:\nImagine three different experts are answering this question.\nAll experts will write down 1 step of their thinking,\nthen share it with the group.\nThen all experts will go on to the next step, etc.\nIf any expert realises they're wrong at any point then they leave.\nThe question is...\nSun (2023) (opens in a new tab) benchmarked the Tree-of-Thought Prompting with large-scale experiments, and introduce PanelGPT --- an idea of prompting with Panel discussions among LLMs.Generate Knowledge PromptingRetrieval Augmented GenerationEnglishLightCopyright © 2023 DAIR.AI", metadata={'language': 'en', 'source': 'http://promptingguide.ai//techniques/tot', 'title': 'Tree of Thoughts (ToT) | Prompt Engineering Guide '}),
 Document(page_content="\n\n\n\n\nğŸŸ¢ What's in a Prompt? | Learn 

In [31]:

import time

index_name = "sparklearn"  # change if desired
from pinecone import ServerlessSpec
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

In [36]:
index = pc.Index(index_name)

from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embeddings)

In [28]:
import re

def clean_text(text):
    # Replace multiple whitespaces (except newlines) with a single space
    text = re.sub(r'(?!\n)\s+', ' ', text)
    
    # Replace multiple newlines with a single newline
    text = re.sub(r'\n+', '\n', text)
    
    # Remove leading and trailing whitespace
    text = text.strip()
    
    return text

# Example usage
test = """
This is some example text.


With multiple newlines.


And    unnecessary    whitespace.   """

cleaned_data = clean_text(test)
print(cleaned_data)

This is some example text.
With multiple newlines.
And unnecessary whitespace.


In [44]:
from langchain.schema import Document

cleaned_docs = [
    Document(
        metadata=doc.metadata,
        page_content=clean_text(doc.page_content)  # Clean the page content
    )
    for doc in docs
]

In [45]:
cleaned_docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/', 'title': "Prompt Engineering | Lil'Log", 'description': 'Prompt Engineering, also known as In-Context Prompting, refers to methods for how to communicate with LLM to steer its behavior for desired outcomes without updating the model weights. It is an empirical science and the effect of prompt engineering methods can vary a lot among models, thus requiring heavy experimentation and heuristics.\nThis post only focuses on prompt engineering for autoregressive language models, so nothing with Cloze tests, image generation or multimodality models. At its core, the goal of prompt engineering is about alignment and model steerability. Check my previous post on controllable text generation.', 'language': 'en', 'text': "\n\n\n\n\n\nPrompt Engineering | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPo

In [46]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(cleaned_page_content))]

vector_store.add_documents(documents=cleaned_docs, ids=uuids)

# vector_store.add_documents(documents=cleaned_page_content)

['3c744f2b-cfae-4e7c-8d8e-d5877c83cd1f',
 '2f416cf9-a55f-48ee-be3e-efd2808edab2',
 '7dea60dd-1c18-45b6-adb2-a70445b4f596',
 '5476216f-60e8-4168-bf52-9256b60eb9d0',
 '34cffbec-ca1c-48d9-a917-7f53ae1a0bcb',
 'e999b3cd-d9e8-4b0f-b38c-437b741ec4f0',
 'd8a6a509-628d-45be-b62c-a5ade53ca1eb',
 '8298becc-303d-4565-90ce-0e825fae7ab6',
 '3057ca7f-1da5-458b-8a89-318cdaea011e',
 '29ccaddb-3191-47cf-ab1a-1c13c522f6b8',
 '129f352f-0586-4eca-b230-991e4c749ead',
 '54e7c9ae-2150-4aad-83d9-28e3f592548b',
 '8b299c7f-54f7-444b-a1f4-0d93368a06ce',
 '8a4b994b-12f5-4f5a-a903-55737c3ca233',
 'fdac2801-2a6e-4102-ac93-34d611a5e499',
 '30b7b88d-aeff-4fc0-82c7-51501c4bcdfd',
 'b0028ddd-9d08-4553-a715-ce14f4405f47',
 '4bd4a3d7-695e-454c-938f-0e18605909a9',
 '58085a33-9fcc-425b-913e-06815ec90e70',
 '772157dc-fb19-4a94-8547-cb7b7fadbe6f',
 '80bb2a5e-d4bb-434d-adc7-abde4f1eb2ea',
 'f6d1db78-768a-48de-8813-7e7696aa22ab',
 '32e75688-ed8e-4a4d-a72c-ef65122565f0',
 '9ebdc2e8-b206-40fe-8d41-f1015c88497a',
 'b4bd51d0-79dd-

In [26]:
learn_index = pc.Index('sparklearn')

learnsearch = PineconeVectorStore(index=learn_index, embedding=embeddings)
learn_retriever = learnsearch.as_retriever(search_kwargs={"k": 8})


In [27]:
learn_retriever.invoke('chain of thoight')

[Document(id='605b4c8d-3132-4718-b78e-865fa29cc34f', metadata={'description': 'Learn how Chain-of-Thought prompting improves AI reasoning by guiding models to explain their thought process. Discover its impact on LLM accuracy and complex tasks.', 'language': 'en', 'source': 'https://learnprompting.org/docs/intermediate/chain_of_thought', 'title': 'Chain-of-Thought Prompting'}, page_content=' steps for decision-making tasks.\n\nHow to Use Chain-of-Thought Prompting\n\xa0CopyChain-of-Thought Prompting TemplateQ: John has 10 apples. He gives away 4 and then receives 5 more. How many apples does he have?A:\nJohn starts with 10 apples.\nHe gives away 4, so 10 - 4 = 6.\nHe then receives 5 more apples, so 6 + 5 = 11.\nFinal Answer: 11\nQ: [Your Question]\nExamples\nHere are two demos illustrating how CoT prompting improves outcomes. The first demo shows GPT-3 (davinci-003) struggling with a word problem without CoT, while the second shows it succeeding using CoT.\nIncorrect Solution (Without 